In [28]:
import pandas as pd
df = pd.read_excel("/content/institutions-and-researchers.xlsx")
df.head()

#Verificar colunas com valores nulos
num_nulos = df.isnull().sum()
print(f'Valores nulos no DataSet: {num_nulos}')

#Verificar valores duplicados
num_duplicadas = df.duplicated().sum()
print(f"Valores Duplicados no Dataset: {num_duplicadas}")

#Corrigindo valore em novo dataset
df_novo = df
df_novo = df_novo.drop_duplicates()
num_duplicads = df_novo.duplicated().sum()
df_novo['Research Name'] = df_novo['Research Name'].fillna("Anomyous")

print(f"Valores Duplicados no novo Dataset: {num_duplicads}")
print("Novo Dataset:")
display(df_novo.iloc[340:360, 0])



Valores nulos no DataSet: Research Name             2
University / Institute    0
Country                   0
dtype: int64
Valores Duplicados no Dataset: 1
Valores Duplicados no novo Dataset: 0
Novo Dataset:


/tmp/ipykernel_729/676714415.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_novo['Research Name'] = df_novo['Research Name'].fillna("Anomyous")


,Research Name
341,Samsul Khabib
342,Yasir S. Alsalamah
343,Bander Saad Albagawi
344,Lisa Babkair
345,Ibrahim Alasqah
346,Talal Ali F Alharbi
347,Anomyous
348,Nada Alqarawi
349,Hana Khaled ALOtheimin
350,Faisal I. Almohaileb


##Normalizando multiplas instituições em uma linha

In [29]:

# Separando multiplas instituições em uma linha
col_nome = "University / Institute"
df_novo[col_nome] = df_novo[col_nome].str.split("/")
df_novo = df_novo.explode(col_nome).reset_index(drop=True)

# Removendo espaços em branco
df_novo[col_nome] = df_novo[col_nome].str.strip()


/tmp/ipykernel_729/1893364840.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_novo[col_nome] = df_novo[col_nome].str.split("/")


##Separando Tabelas, Criando IDs e Relacionamentos

In [30]:
#Criando chaves de ID:
df_novo['researcher_id'] = pd.factorize(df_novo['Research Name'])[0] #cada pesquisador tem um ID único
agrupamento = df_novo[['University / Institute', 'Country']].apply(tuple, axis=1) #considerando combinações únicas de instituição e pais
df_novo['institution_id'] = pd.factorize(agrupamento)[0] #filias em diferentes paises são tidas como instituições diferentes
print("Novo Dataset:")
display(df_novo)

#Criando Tabela de Instituições
df_institutions = df_novo[['University / Institute', 'Country', 'institution_id']].copy()
print("Tabela Instituições:")
df_institutions = df_institutions.drop_duplicates()
display(df_institutions)

#Criando tabela de Pesquisadores
df_researchers = df_novo[['Research Name', 'researcher_id', 'institution_id']].copy()
df_researchers = df_researchers.drop_duplicates()
print("Tabela Pesquisadores:")
display(df_researchers)



Novo Dataset:


,Research Name,University / Institute,Country,researcher_id,institution_id
0,Danilo Rodrigues Pereira,"University of Campinas (UNICAMP), Brazil",Brazil,0,0
1,Simone Appenzeller,"University of Campinas (UNICAMP), Brazil",Brazil,1,0
2,André Franceschi de Angelis,"University of Campinas (UNICAMP), Brazil",Brazil,2,0
3,Thais Rocha,"University of Campinas (UNICAMP), Brazil",Brazil,3,0
4,Danielle F. dos Santos Dias,"University of Campinas (UNICAMP), Brazil",Brazil,4,0
...,...,...,...,...,...
394,Imam Much Ibnu Subroto,"Universitas Islam Sultan Agung, Indonesia",Indonesia,380,248
395,Ir. Suryani Alifah,"Universitas Islam Sultan Agung, Indonesia",Indonesia,381,248
396,Ira Puspitasari,Universitas Airlangga - Indonesia,Indonesia,382,249
397,Barry Nuqoba,Universitas Airlangga - Indonesia,Indonesia,383,249


Tabela Instituições:


,University / Institute,Country,institution_id
0,"University of Campinas (UNICAMP), Brazil",Brazil,0
7,"Federal University of Southern Bahia (UFSB), B...",Brazil,1
8,Paulista University (UNIP) - Brazil,Brazil,2
10,Federal University of Maranhão (UFMA) - Brazil,Brazil,3
11,Federal University of Rio Grande do Sul (UFRGS),Brazil,4
...,...,...,...
389,"STKIP PGRI Trenggalek, Indonesia",Indonesia,245
390,"Czech University of Life Sciences Prague, Czec...",Czech Republic,246
391,Mindanao State University - Iligan Institute o...,Philippines,247
393,"Universitas Islam Sultan Agung, Indonesia",Indonesia,248


Tabela Pesquisadores:


,Research Name,researcher_id,institution_id
0,Danilo Rodrigues Pereira,0,0
1,Simone Appenzeller,1,0
2,André Franceschi de Angelis,2,0
3,Thais Rocha,3,0
4,Danielle F. dos Santos Dias,4,0
...,...,...,...
394,Imam Much Ibnu Subroto,380,248
395,Ir. Suryani Alifah,381,248
396,Ira Puspitasari,382,249
397,Barry Nuqoba,383,249


In [31]:
#Exportando novas tabelas
df_institutions.to_csv('institutions.csv', index=False, encoding='utf-8')
df_researchers.to_csv('researchers.csv', index=False, encoding='utf-8')